In [3]:
import awswrangler as wr
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import boto3

In [4]:
# Configurando a região padrão
boto3.setup_default_session(region_name = "us-east-1")

GLUE_DATABASE = "classicmodels_analytics"
BUCKET_NAME = "classicmodels-datalake-pedro-coterli"
BASE_PATH = f"s3://{BUCKET_NAME}/output"

# Criando o Database no AWS Glue
databases = wr.catalog.databases().values

if GLUE_DATABASE not in databases:
    wr.catalog.create_database(GLUE_DATABASE)
    print(f"Database '{GLUE_DATABASE}' criado no AWS Glue.")

# Registrando as tabelas Parquet no Athena/Glue
tables = ["dim_customers", "dim_products", "dim_dates", "dim_countries", "fact_orders"]

for table in tables:
    wr.s3.store_parquet_metadata(
        path = f"{BASE_PATH}/{table}/",
        database = GLUE_DATABASE,
        table = table,
        dataset = True,
        mode = "overwrite",
    )

print("Tabelas catalogadas com sucesso! O Athena já pode consultar os dados.")

Database 'classicmodels_analytics' criado no AWS Glue.
Tabelas catalogadas com sucesso! O Athena já pode consultar os dados.


In [5]:
exploratory_query = """
    SELECT
        product_id,
        product_name,
        product_line,
        product_vendor
    FROM dim_products
    LIMIT 20
"""

# Executando a query no Athena
df_products = wr.athena.read_sql_query(sql = exploratory_query, database = GLUE_DATABASE)

print("--- Explorando dim_products ---")
display(df_products.head())

--- Explorando dim_products ---


,product_id,product_name,product_line,product_vendor
0,S18_1889,1948 Porsche 356-A Roadster,Classic Cars,Gearbox Collectibles
1,S700_1938,The Mayflower,Ships,Studio M Art Models
2,S32_1374,1997 BMW F650 ST,Motorcycles,Exoto Designs
3,S18_2432,1926 Ford Fire Engine,Trucks and Buses,Carousel DieCast Legends
4,S24_2011,18th century schooner,Ships,Carousel DieCast Legends


In [6]:
country_sales_query = """
    SELECT
        dim_countries.country,
        SUM(fact_orders.sales_amount) AS total_sales
    FROM fact_orders
    JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
    GROUP BY dim_countries.country
    ORDER BY total_sales DESC
    LIMIT 10
"""

df_country_sales = wr.athena.read_sql_query(sql = country_sales_query, database = GLUE_DATABASE)

print("--- Top 10 países por vendas ---")
display(df_country_sales)

--- Top 10 países por vendas ---


,country,total_sales
0,USA,3273280.05
1,Spain,1099389.09
2,France,1007374.02
3,Australia,562582.59
4,New Zealand,476847.01
5,UK,436947.44
6,Italy,360616.81
7,Finland,295149.35
8,Singapore,263997.78
9,Denmark,218994.92


In [ ]:
detailing_query = """
    SELECT
        dim_dates.full_date,
        dim_products.product_line,
        dim_products.product_name,
        dim_countries.country,
        SUM(fact_orders.sales_amount) AS total_sales
    FROM fact_orders
    JOIN dim_products ON fact_orders.product_id = dim_products.product_id
    JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
    JOIN dim_dates ON fact_orders.order_date_key = dim_dates.date_key
    GROUP BY
        dim_dates.full_date,
        dim_products.product_line,
        dim_products.product_name,
        dim_countries.country
"""

# Extraindo a base analítica
df_analytics = wr.athena.read_sql_query(sql = detailing_query, database = GLUE_DATABASE)

# Convertendo a coluna de data para o tipo datetime do Pandas
df_analytics["full_date"] = pd.to_datetime(df_analytics["full_date"])

# Corrigindo "Norway" duplicada
df_analytics.loc[df_analytics["country"] == "Norway  ", "country"] = "Norway"

print(f"Base analítica carregada com {len(df_analytics)} registros.")
display(df_analytics.head())

Base analítica carregada com 2996 registros.


,full_date,product_line,product_name,country,total_sales
0,2004-02-12,Classic Cars,1969 Corvair Monza,Ireland,4532.40
1,2004-09-09,Ships,The Mayflower,Italy,2260.55
2,2004-11-17,Vintage Cars,1917 Grand Touring Sedan,USA,6806.80
3,2004-08-02,Classic Cars,1970 Plymouth Hemi Cuda,USA,2577.54
4,2004-11-19,Classic Cars,1958 Chevy Corvette Limited Edition,USA,1085.04


In [19]:
# Definindo as opções de filtros
country_options = ["Todos"] + sorted(df_analytics["country"].dropna().unique().tolist())
lines_options = ["Todos"] + sorted(df_analytics["product_line"].dropna().unique().tolist())

# Obtendo os limites de data para os widgets
data_min = df_analytics["full_date"].min().date()
data_max = df_analytics["full_date"].max().date()

# Criando os widgets de interface
start_date_widget = widgets.DatePicker(description = "Data inicial", value = data_min)
end_date_widget = widgets.DatePicker(description = "Data final", value = data_max)
country_widget = widgets.Dropdown(options = country_options, value = "Todos", description = "País:")
product_line_widget = widgets.Dropdown(
    options = lines_options,
    value = "Todos",
    description = "Linha de produto:",
    style = {"description_width": "initial"}
)
top_n_widget = widgets.IntSlider(
    min = 1,
    max = 15,
    value = 5,
    description = "Top N produtos",
    style = {"description_width": "initial"}
)

# Função que será chamada sempre que um filtro for alterado
def update_dashboard(start_date, end_date, country, product_line, top_n):
    # Filtrando os dados exibidos
    df_filtered = df_analytics.copy()

    if start_date:
        df_filtered = df_filtered[df_filtered["full_date"].dt.date >= start_date]
    if end_date:
        df_filtered = df_filtered[df_filtered["full_date"].dt.date <= end_date]
    if country != "Todos":
        df_filtered = df_filtered[df_filtered["country"] == country]
    if product_line != "Todos":
        df_filtered = df_filtered[df_filtered["product_line"] == product_line]

    # Agregando e ranqueando
    df_agg = df_filtered.groupby("product_name", as_index = False)["total_sales"].sum()
    df_top_n = df_agg.sort_values(by = "total_sales", ascending = False).head(top_n)

    # Plotando o gráfico
    plt.figure(figsize = (10, 6))

    if df_top_n.empty:
        print("Nenhum dado retornado para os filtros selecionados.")
        return
    
    sns.barplot(data = df_top_n, x = "total_sales", y = "product_name", color = "#1f77b4")

    plt.title(f"Top {top_n} produtos por vendas", fontsize = 14, pad = 15)
    plt.xlabel("Vendas totais ($)", fontsize = 12)
    plt.ylabel("Produto", fontsize = 12)
    plt.grid(axis = "x", linestyle = "--", alpha = 0.7)
    plt.tight_layout()
    plt.show()

# Conectando os widgets à função de atualização
ui_dates = widgets.HBox([start_date_widget, end_date_widget])
ui_filters = widgets.HBox([country_widget, product_line_widget])
ui_slider = widgets.HBox([top_n_widget])

out = widgets.interactive_output(
    update_dashboard,
    {
        "start_date": start_date_widget,
        "end_date": end_date_widget,
        "country": country_widget,
        "product_line": product_line_widget,
        "top_n": top_n_widget,
    }
)

display(ui_dates, ui_filters, ui_slider, out)

Output()